In [0]:
dbutils.library.restartPython()

1. Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
-- Create tables in own database
USE gr5069.gbm2118;

In [0]:
%sql
-- Table for storing predictions of positionOrder from results dataset (Model 1)
CREATE TABLE results_ridge_reg (
resultId INT,
raceId INT,
driverId INT,
constructorId INT,
grid INT,
positionOrder INT,
pred DOUBLE,
residual DOUBLE
);

In [0]:
%sql
-- Table for storing predictions of point scored from results dataset (Model 2)
CREATE TABLE results_random_forest (
resultId INT,
raceId INT,
driverId INT,
constructorId INT,
grid INT,
points DOUBLE,
y_pred DOUBLE,
residual DOUBLE
);

2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

## Model 1: Ridge Regression

I will predict the numerical order of finish (positionOrder) using ridge regression. My features are: resultId, raceId, driverId, constructorId and grid.

In [0]:
# Import important libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark
!pip install statsmodels
import statsmodels.api as sm
import mlflow.sklearn

In [0]:
# Load the results data
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

# Inspect the data
display(df_results)

In [0]:
# Convert resultId, raceId, driverId, constructorId and grid to integers
df_results = df_results.withColumn("resultId", df_results["resultId"].cast("int"))
df_results = df_results.withColumn("raceId", df_results["raceId"].cast("int"))
df_results = df_results.withColumn("driverId", df_results["driverId"].cast("int"))
df_results = df_results.withColumn("constructorId", df_results["constructorId"].cast("int"))
df_results = df_results.withColumn("grid", df_results["grid"].cast("int"))

# Inspect the data
display(df_results)

In [0]:
# Change variable names to X, y to prepare for train/test split
df = df_results.toPandas()
y = df["positionOrder"]
X = df[["resultId", "raceId", "driverId", "constructorId", "grid"]]

# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Build a linear regression model

In [0]:
# Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

params = {"n_estimators": 200, "max_depth": 5, "random_state": 42}
lr = LinearRegression(**params)
lr.fit(X_train, y_train)
predictions = lr.predict(X_test)

# Create metrics to evaluate model
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print("  mse: {}".format(mse))
print("  mae: {}".format(mae))
print("  R2: {}".format(r2))

# Create feature importance
feature_importances = pd.DataFrame(rf.feature_importances_, index=X_train.columns, columns=["importance"])
feature_importances.sort_values("importance", ascending=False)
print(feature_importances)

# Create plot
fig, ax = plt.subplots()

sns.residplot(x=predictions, y=y_test.astype(float))
plt.xlabel("Predicted values for Driver Points")
plt.ylabel("Residual")
plt.title("Residual Plot")